# nb01 - Clean: LA County Medi-Cal market share (with statewide context)

Turns the raw enrollment file into two analysis files:

1. `la_market_share_clean.csv` - Los Angeles County, monthly, one row per plan (L.A. Care, Health Net, Kaiser), medical managed care only.
2. `ca_county_enrollment_clean.csv` - every California county, total Medi-Cal medical managed care enrollment, for the statewide choropleth and the "L.A. Care is the largest plan in the state" context.

**Three cleaning problems handled here (documented for the Connect and Transform section of the post):**
- The `Plan Type` label changed mid-series: `Two-Plan` (2007 to Jul 2018) became `Local Initiative (2 Plan)` + `Commercial Plan (2 Plan)` (Aug 2018 on). The eras do not overlap within a month, so the series joins cleanly.
- Plan names have ALL CAPS and renamed variants (`LA CARE` -> `L.A. Care Health Plan/Los Angeles`; `Health Net / LA` -> `Health Net Community Solutions/Los Angeles`). Merged with a grouping map.
- Six non-medical product lines (Dental, PACE, SCAN, Cal MediConnect, PCCM, Special Project) are filtered out; market share is only meaningful over the medical lines.
- Suppressed small counts become nulls, not zeros.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path('..') / 'data'
raw_files = sorted((DATA / 'raw').glob('medi_cal_mc_enrollment_raw_*.csv'))
RAW = raw_files[-1]
print('reading', RAW.name)

df = pd.read_csv(RAW, dtype=str)
df.columns = [c.strip() for c in df.columns]
for c in ['Enrollment Month', 'Plan Type', 'County', 'Plan Name']:
    df[c] = df[c].str.strip()
# suppressed counts -> null, not zero
df['Enrollees'] = pd.to_numeric(df['Count of Enrollees'].str.replace(',', '').str.strip(), errors='coerce')
df['Year'] = df['Enrollment Month'].str[:4].astype(int)
print('raw shape:', df.shape, '| months', df['Enrollment Month'].min(), '..', df['Enrollment Month'].max())
print('suppressed (null) rows:', df['Enrollees'].isna().sum())

reading medi_cal_mc_enrollment_raw_2026-07-22.csv
raw shape: (31178, 10) | months 2007-01 .. 2026-06
suppressed (null) rows: 689


## 1. Keep only the medical managed care lines

Medi-Cal members can hold a medical plan AND a separate dental or specialty plan, so summing every plan type double counts. Market share is computed over the medical models only.

In [2]:
MEDICAL_TYPES = {
    'Two-Plan',                        # 2007 - Jul 2018 (old label)
    'Local Initiative (2 Plan)',       # Aug 2018 on (public plan, e.g. L.A. Care)
    'Commercial Plan (2 Plan)',        # Aug 2018 on (commercial plan, e.g. Health Net)
    'County Organized Health Systems', # COHS counties
    'Geographic Managed Care',         # GMC (Sacramento, San Diego)
    'Single', 'Regional Model',        # other medical models
    'Prepaid Health Plan',             # historical medical
}
EXCLUDE = {'Dental', 'PACE', 'SCAN', 'Special Project',
           'Primary Care Case Management', 'Primary Care Case Mgmt', 'Cal MediConnect'}

med = df[df['Plan Type'].isin(MEDICAL_TYPES)].copy()
print('medical rows:', len(med), 'of', len(df))
print('excluded plan types present:', sorted(set(df['Plan Type']) & EXCLUDE))
print('kept plan types:', sorted(med['Plan Type'].unique()))

medical rows: 11992 of 31178
excluded plan types present: ['Cal MediConnect', 'Dental', 'PACE', 'Primary Care Case Management', 'Primary Care Case Mgmt', 'SCAN', 'Special Project']
kept plan types: ['Commercial Plan (2 Plan)', 'County Organized Health Systems', 'Geographic Managed Care', 'Local Initiative (2 Plan)', 'Prepaid Health Plan', 'Regional Model', 'Single', 'Two-Plan']


## 2. Normalize the plan model across the two eras

`Two-Plan` predates the split into Local Initiative and Commercial. A single `Plan Model` field, derived by plan, makes the 19-year series continuous.

In [3]:
def plan_brand(name):
    n = name.upper()
    if 'CARE' in n and ('LA CARE' in n or 'L.A. CARE' in n):
        return 'L.A. Care'
    if 'HEALTH NET' in n:
        return 'Health Net'
    if 'KAISER' in n:
        return 'Kaiser Permanente'
    return name  # statewide plans keep their name

med['Brand'] = med['Plan Name'].map(plan_brand)

# Plan Model: normalize the old 'Two-Plan' rows into Local Initiative / Commercial by brand
def plan_model(row):
    pt = row['Plan Type']
    if pt == 'Two-Plan':
        return 'Local Initiative' if row['Brand'] == 'L.A. Care' else 'Commercial Plan'
    return pt.replace(' (2 Plan)', '')

med['Plan Model'] = med.apply(plan_model, axis=1)
print(med[med.County == 'Los Angeles'].groupby(['Plan Model', 'Brand']).size().to_string())

Plan Model        Brand            
Commercial Plan   Health Net           234
                  Kaiser Permanente     30
Local Initiative  L.A. Care            234


## 3. Output 1 - Los Angeles County market share (monthly, by plan)

One row per month per brand. Share is intentionally NOT precomputed here: it is a FIXED LOD in Tableau (plan enrollees divided by the month's county total), which is one of the teaching points. It is added here only for the parity notebook to check against.

In [4]:
la = med[med.County == 'Los Angeles'].copy()
la_plan = (la.groupby(['Enrollment Month', 'Year', 'Brand'], as_index=False)
             .Enrollees.sum())

# monthly county total (medical), for the share denominator and for parity
la_plan['County Total'] = la_plan.groupby('Enrollment Month').Enrollees.transform('sum')
la_plan['Share'] = (la_plan.Enrollees / la_plan['County Total']).round(4)

la_plan = la_plan.sort_values(['Enrollment Month', 'Enrollees'], ascending=[True, False])
la_plan.to_csv(DATA / 'la_market_share_clean.csv', index=False)
print('wrote la_market_share_clean.csv', la_plan.shape)

latest = la_plan['Enrollment Month'].max()
print(f'\n=== LA County medical managed care market, {latest} ===')
cur = la_plan[la_plan['Enrollment Month'] == latest]
for r in cur.itertuples():
    print(f'{r.Share*100:5.1f}%  {int(r.Enrollees):>9,}  {r.Brand}')
print(f'TOTAL {int(cur["County Total"].iloc[0]):,}')

wrote la_market_share_clean.csv (498, 6)

=== LA County medical managed care market, 2026-06 ===
 59.7%  2,102,545  L.A. Care
 30.7%  1,079,978  Health Net
  9.6%    339,151  Kaiser Permanente
TOTAL 3,521,674


## 4. Output 2 - statewide county totals (for the choropleth and the state context)

Total Medi-Cal medical managed care enrollment by county, latest month, plus the largest single plan statewide.

In [5]:
latest = med['Enrollment Month'].max()
cur = med[med['Enrollment Month'] == latest]

county = (cur.groupby('County', as_index=False).Enrollees.sum()
             .rename(columns={'Enrollees': 'Medi-Cal MC Enrollees'})
             .sort_values('Medi-Cal MC Enrollees', ascending=False))
county['Enrollment Month'] = latest
county.to_csv(DATA / 'ca_county_enrollment_clean.csv', index=False)
print('wrote ca_county_enrollment_clean.csv', county.shape)
print(f'\n=== top 8 counties by Medi-Cal managed care enrollment, {latest} ===')
print(county.head(8).assign(**{'Medi-Cal MC Enrollees': county['Medi-Cal MC Enrollees'].map(lambda v: f'{int(v):,}')}).to_string(index=False))

# is L.A. Care the largest single plan in the state?
by_plan = cur.groupby('Plan Name').Enrollees.sum().sort_values(ascending=False)
print(f'\n=== largest single medical plans in California, {latest} ===')
print(by_plan.head(6).map(lambda v: f'{int(v):,}').to_string())

wrote ca_county_enrollment_clean.csv (58, 3)

=== top 8 counties by Medi-Cal managed care enrollment, 2026-06 ===
        County Medi-Cal MC Enrollees Enrollment Month
   Los Angeles             3,521,674          2026-06
San Bernardino               866,361          2026-06
     Riverside               862,488          2026-06
        Orange               855,944          2026-06
     San Diego               821,390          2026-06
    Sacramento               560,275          2026-06
        Fresno               473,382          2026-06
          Kern               448,377          2026-06

=== largest single medical plans in California, 2026-06 ===
Plan Name
L.A. Care Health Plan/Los Angeles             2,102,545
Health Net Community Solutions/Los Angeles    1,079,978
CalOptima/Orange                                783,907
Inland Empire Health Plan/San Bernardino        689,551
Inland Empire Health Plan/Riverside             686,144
Kern Health Systems/Kern                        3

## 5. QA - the numbers the post will use

In [6]:
print('LA market share over time (June of selected years):')
for yr in [2008, 2012, 2016, 2020, 2024, 2026]:
    m = f'{yr}-06'
    sub = la_plan[la_plan['Enrollment Month'] == m]
    if len(sub):
        lac = sub[sub.Brand == 'L.A. Care']
        tot = sub['County Total'].iloc[0]
        share = lac.Share.iloc[0]*100 if len(lac) else 0
        print(f'  {m}: market {int(tot):>9,} | L.A. Care {share:4.1f}%')

# sanity: shares within a month sum to 1
chk = la_plan.groupby('Enrollment Month').Share.sum()
print('\nshare sums per month min/max:', round(chk.min(), 4), round(chk.max(), 4), '(should be ~1.0)')

LA market share over time (June of selected years):
  2008-06: market 1,162,003 | L.A. Care 63.8%
  2012-06: market 1,485,079 | L.A. Care 67.0%
  2016-06: market 3,043,673 | L.A. Care 65.5%
  2020-06: market 2,999,743 | L.A. Care 68.7%
  2024-06: market 3,834,674 | L.A. Care 61.5%
  2026-06: market 3,521,674 | L.A. Care 59.7%

share sums per month min/max: 0.9999 1.0001 (should be ~1.0)
